In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

# PubMed API URL
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

# Function to search PubMed for prostate cancer papers
def search_pubmed(query, max_results=50):
    search_url = f"{BASE_URL}esearch.fcgi?db=pubmed&term={query}&retmax={max_results}&retmode=xml"
    response = requests.get(search_url)
    root = ET.fromstring(response.text)
    pmids = [id_elem.text for id_elem in root.findall(".//Id")]
    return pmids

# Function to fetch paper details
def fetch_paper_details(pmids):
    details_url = f"{BASE_URL}efetch.fcgi?db=pubmed&id={','.join(pmids)}&retmode=xml"
    response = requests.get(details_url)
    root = ET.fromstring(response.text)

    papers = []
    for article in root.findall(".//PubmedArticle"):
        pmid = article.find(".//PMID").text
        title = article.find(".//ArticleTitle").text if article.find(".//ArticleTitle") is not None else "N/A"
        abstract = article.find(".//AbstractText").text if article.find(".//AbstractText") is not None else "N/A"
        papers.append({"PMID": pmid, "Title": title, "Abstract": abstract})

    return papers

# Main function to run search and fetch details
def main():
    query = "prostate cancer biomarkers"
    print(f"Searching PubMed for: {query}")
    pmids = search_pubmed(query)
    print(f"Found {len(pmids)} papers. Fetching details...")
    papers = fetch_paper_details(pmids)

    # Save to CSV
    df = pd.DataFrame(papers)
    df.to_csv("pubmed_prostate_cancer_papers.csv", index=False)
    print("Saved results to pubmed_prostate_cancer_papers.csv")

if __name__ == "__main__":
    main()

Searching PubMed for: prostate cancer biomarkers
Found 50 papers. Fetching details...
Saved results to pubmed_prostate_cancer_papers.csv


In [ ]:
!pip install biopython
from Bio import Entrez
import pandas as pd

# Set email (required by NCBI Entrez)
Entrez.email = "your_email@example.com"

# GEO dataset ID
geo_id = "GSE45016"

# Fetch GEO dataset
handle = Entrez.efetch(db="gds", id=geo_id, rettype="full", retmode="text")
geo_data = handle.read()
handle.close()

# Save raw data to a text file
with open("GSE45016_raw.txt", "w") as file:
    file.write(geo_data)

print("GSE45016 dataset downloaded and saved as GSE45016_raw.txt")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 25.8 MB/s eta 0:00:00
GSE45016 dataset downloaded and saved as GSE45016_raw.txt


In [ ]:
!pip install GEOparse

import GEOparse
import pandas as pd

# Download and parse GEO dataset
geo_id = "GSE45016"
gse = GEOparse.get_GEO(geo=geo_id, destdir="./")

# Extract expression data
expression_data = gse.pivot_samples('VALUE')

# Save expression data to CSV
expression_data.to_csv("GSE45016_expression.csv")
expression_data.to_csv("/content/sample_data/GSE45016_expression.csv")  # Save for further processing

print("GSE45016 expression data saved successfully!")



19-Mar-2025 18:59:55 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
19-Mar-2025 18:59:55 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
19-Mar-2025 18:59:55 INFO GEOparse - Parsing ./GSE45016_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE45016_family.soft.gz: 
19-Mar-2025 18:59:55 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
19-Mar-2025 18:59:55 DEBUG GEOparse - SERIES: GSE45016
DEBUG:GEOparse:SERIES: GSE45016
19-Mar-2025 18:59:55 DEBUG GEOparse - PLATFORM: GPL570
DEBUG:GEOparse:PLATFORM: GPL570
/usr/local/lib/python3.11/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
19-Mar-2025 18:59:57 DEBUG GEOparse - SAMPLE: GSM1095876
DEBUG:GEOparse:SAMPLE: GSM1095876
19-Mar-2025 18:59:57 DEBUG GEOpars

GSE45016 expression data saved successfully!


In [ ]:
import pandas as pd

# Load the dataset
file_path = "/content/GSE45016_expression.csv"
df = pd.read_csv(file_path, index_col=0)

# Display basic information
print(df.head())  # Preview first few rows
print(df.info())  # Get summary of data


           GSM1095876  GSM1095877  GSM1095878  GSM1095879  GSM1095880  \
ID_REF                                                                  
1007_s_at      3333.3      3195.5      2989.7      3514.7      2032.7   
1053_at         568.8       393.5       107.5       319.8       328.8   
117_at           87.0        79.8       146.1       101.5       169.9   
121_at          466.8       883.3       746.9       351.2       428.4   
1255_g_at        31.5         8.7        21.7        54.7        52.1   

           GSM1095881  GSM1095882  GSM1095883  GSM1095884  GSM1095885  \
ID_REF                                                                  
1007_s_at      5985.0      2660.4      3362.6      3190.5      2786.8   
1053_at         358.4       448.9       400.0       417.8       308.8   
117_at           79.4        49.0        70.9        88.7        39.9   
121_at         1242.7       871.6       362.8       454.7       530.7   
1255_g_at        15.5        51.0         5.0     

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "/content/GSE45016_expression.csv"
df = pd.read_csv(file_path, index_col=0)

# Convert expression values to numeric (force non-numeric to NaN)
df.iloc[:, 1:] = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

# Drop rows with too many NaN values (optional)
df.dropna(axis=0, thresh=int(0.5 * df.shape[1]), inplace=True)

# Define cutoff for significance
logfc_cutoff = 1.0

# Calculate Log2 Fold Change (LogFC) using mean expression across samples
df["LogFC"] = np.log2(df.iloc[:, 1:].mean(axis=1) / df.iloc[:, 1:].median(axis=1))

# Identify upregulated and downregulated genes
df["Regulation"] = df["LogFC"].apply(lambda x: "Upregulated" if x > logfc_cutoff else "Downregulated" if x < -logfc_cutoff else "Not Significant")

# Save results
df.to_csv("/content/sample_data/GSE45016_DEGs.csv")

# Summary counts
print(df["Regulation"].value_counts())



Regulation
Not Significant    50067
Upregulated         4608
Name: count, dtype: int64
